# 003 · Command-Line Arguments with argparse

> **Standard Library › System & CLI** · Python 3.14

`sys.argv` gives you raw strings; `argparse` turns them into typed, validated values, generates `--help` text, and prints friendly error messages. It is Python's built-in answer to what `System.CommandLine` does in .NET.

**In this notebook you will learn:**

- how to declare positional and optional arguments, with types, defaults and choices
- flags (`store_true`, `BooleanOptionalAction`), repeated options and `nargs`
- the generated help and usage text
- subcommands (`git commit`-style) with `add_subparsers`
- how to drive and test a parser from a notebook with `parse_args([...])`, including error handling

## .NET mental model

| .NET (`System.CommandLine`) | Python (`argparse`) |
|---|---|
| `RootCommand` | `ArgumentParser` |
| `Argument<T>` | positional: `add_argument("name", type=T)` |
| `Option<T>` | optional: `add_argument("--name", type=T)` |
| `Option<bool>` | `action="store_true"` / `BooleanOptionalAction` |
| `Command` + `root.Subcommands.Add(...)` | `add_subparsers()` + `add_parser(...)` |
| `ParseResult` + `GetValue(option)` | `Namespace` with attributes: `args.limit` |
| `AcceptOnlyFromAmong(...)` | `choices=[...]` |
| `ParseResult.Errors` | error message on stderr + `SystemExit(2)` |

The big difference: `System.CommandLine` is a **NuGet package**, whereas `argparse` ships with Python. (Popular third-party alternatives in Python are `click` and `typer`; `argparse` is all you need for most tools.)

## A first parser

You create a parser, declare arguments, then call `parse_args()`. In a script, `parse_args()` with no arguments reads `sys.argv[1:]`. In a notebook (and in tests) we pass a **list of strings** instead — exactly what the shell would have produced.

```csharp
// C# — System.CommandLine 2.0
Argument<string> inputArg = new("input") { Description = "Input file" };
Option<int> limitOption = new("--limit", "-n") { Description = "Max rows", DefaultValueFactory = _ => 10 };

RootCommand root = new("Process orders.");
root.Arguments.Add(inputArg);
root.Options.Add(limitOption);

ParseResult result = root.Parse(args);
Console.WriteLine($"{result.GetValue(inputArg)} {result.GetValue(limitOption)}");
```

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="orders", description="Process orders.", color=False)
parser.add_argument("input", help="input file")                                   # positional
parser.add_argument("-n", "--limit", type=int, default=10, help="max rows")       # optional

args = parser.parse_args(["orders.csv", "--limit", "5"])
print(args)                       # a Namespace — a simple attribute bag
print(args.input, args.limit)     # orders.csv 5  — limit is an int, not "5"
print(type(args.limit).__name__)

`prog="orders"` sets the program name shown in help (otherwise it is derived from `sys.argv[0]`, which in a notebook is the kernel launcher).

`color=False` is new in Python 3.14: help and error messages are now coloured when the output supports it, and the Jupyter kernel forces colour on (it sets `FORCE_COLOR=1`). We switch it off so the captured text stays plain. In a terminal, just leave the default.

## Positional vs optional arguments

- Names **without** a leading `-` are **positional** and required by default.
- Names **with** `-`/`--` are **optional**; they default to `None` unless you give a `default`.
- The attribute name comes from the long option, with dashes turned into underscores: `--dry-run` → `args.dry_run`.
- Short and long forms can be combined: `-n 5`, `--limit 5`, `--limit=5` and even `-n5` all work.

In [ ]:
parser = argparse.ArgumentParser(prog="copy", color=False)
parser.add_argument("source")
parser.add_argument("destination")
parser.add_argument("--dry-run", action="store_true")
parser.add_argument("--retries", type=int)          # no default → None

print(parser.parse_args(["a.txt", "b.txt"]))
print(parser.parse_args(["--dry-run", "a.txt", "b.txt", "--retries=3"]))   # options can go anywhere

## Types, defaults and choices

`type=` is any callable that takes a string: `int`, `float`, `pathlib.Path`, or your own function. `choices=` restricts the allowed values. `required=True` makes an *option* mandatory (generally better to use a positional for that).

In [ ]:
from pathlib import Path

def positive_int(text: str) -> int:
    value = int(text)                      # ValueError → "invalid positive_int value"
    if value <= 0:
        raise argparse.ArgumentTypeError(f"{value} is not > 0")
    return value

parser = argparse.ArgumentParser(prog="export", color=False)
parser.add_argument("--out", type=Path, default=Path("report.csv"))
parser.add_argument("--format", choices=["csv", "json"], default="csv")
parser.add_argument("--workers", type=positive_int, default=4)
parser.add_argument("--region", required=True)

args = parser.parse_args(["--region", "eu", "--format", "json", "--out", "data/out.json"])
print(repr(args.out), args.format, args.workers, args.region)

## Flags: `store_true`, `store_false` and `BooleanOptionalAction`

A switch that takes no value uses `action="store_true"` (default `False`). For a flag that can be turned on *and* off explicitly, `BooleanOptionalAction` generates both `--cache` and `--no-cache`:

In [ ]:
parser = argparse.ArgumentParser(prog="build", color=False)
parser.add_argument("-v", "--verbose", action="store_true")
parser.add_argument("--cache", action=argparse.BooleanOptionalAction, default=True)

print(parser.parse_args([]))                        # verbose=False, cache=True
print(parser.parse_args(["-v", "--no-cache"]))      # verbose=True,  cache=False

## Multiple values: `nargs`, `append` and `count`

| Setting | Meaning | Example |
|---|---|---|
| `nargs="?"` | 0 or 1 values | optional positional |
| `nargs="*"` | 0 or more → list | `files*` |
| `nargs="+"` | 1 or more → list | `--ids 1 2 3` |
| `nargs=2` | exactly 2 → list | `--size 800 600` |
| `action="append"` | repeatable option → list | `--tag a --tag b` |
| `action="count"` | count occurrences | `-vvv` → 3 |

In [ ]:
parser = argparse.ArgumentParser(prog="tool", color=False)
parser.add_argument("files", nargs="*")
parser.add_argument("--ids", nargs="+", type=int)
parser.add_argument("--size", nargs=2, type=int, metavar=("WIDTH", "HEIGHT"))
parser.add_argument("--tag", action="append", default=[])
parser.add_argument("-v", action="count", default=0, dest="verbosity")

args = parser.parse_args(["a.txt", "b.txt", "--ids", "1", "2", "--size", "800", "600",
                          "--tag", "x", "--tag", "y", "-vvv"])
print(args.files, args.ids, args.size, args.tag, args.verbosity)

⚠️ **Gotcha:** with `action="append"`, a non-empty default is **not replaced** by what the user passes — the user's values are appended after it. Use `default=None` and fill in the fallback afterwards if you want "replace" semantics.

In [ ]:
appender = argparse.ArgumentParser(prog="tags", color=False)
appender.add_argument("--tag", action="append", default=["base"])
print(appender.parse_args(["--tag", "x"]).tag)     # ['base', 'x'] — not ['x']!

## Generated help

`argparse` builds usage and help text from your declarations. `%(default)s` inside a help string inserts the default value. `print_help()` shows it without exiting (running the script with `-h` prints it and exits with code 0).

In [ ]:
parser = argparse.ArgumentParser(
    prog="orders",
    description="Summarise an orders file.",
    epilog="Example: orders data.csv --limit 20 --format json",
    color=False,
)
parser.add_argument("input", help="CSV file to read")
parser.add_argument("-n", "--limit", type=int, default=10, help="max rows (default: %(default)s)")
parser.add_argument("--format", choices=["table", "json"], default="table", help="output format")
parser.add_argument("--cache", action=argparse.BooleanOptionalAction, default=True, help="use the cache")
parser.add_argument("-v", "--verbose", action="store_true", help="chatty output")

parser.print_help()

## Parse errors raise `SystemExit`

When the input is invalid, `argparse` prints the usage plus an error message to **stderr** and calls `sys.exit(2)`. In a script that is exactly what you want. In a notebook or a test, catch `SystemExit` — and capture stderr with `contextlib.redirect_stderr` to inspect the message:

In [ ]:
import contextlib
import io

def try_parse(parser: argparse.ArgumentParser, argv: list[str]) -> argparse.Namespace | None:
    captured = io.StringIO()
    try:
        with contextlib.redirect_stderr(captured):
            return parser.parse_args(argv)
    except SystemExit as e:
        last_line = captured.getvalue().strip().splitlines()[-1]
        print(f"SystemExit({e.code}) → {last_line}")
        return None

try_parse(parser, [])                                  # missing positional
try_parse(parser, ["data.csv", "--limit", "many"])     # wrong type
try_parse(parser, ["data.csv", "--format", "xml"])     # invalid choice
try_parse(parser, ["data.csv", "--colour"])            # unknown option
print(try_parse(parser, ["data.csv", "-n", "3"]))      # valid → a Namespace

If you would rather get an exception than an exit, construct the parser with `exit_on_error=False`; most errors then raise `argparse.ArgumentError`:

In [ ]:
strict = argparse.ArgumentParser(prog="strict", exit_on_error=False, color=False)
strict.add_argument("--limit", type=int)

try:
    strict.parse_args(["--limit", "many"])
except argparse.ArgumentError as e:
    print(f"{type(e).__name__}: {e}")

**Tip:** Python 3.14 added `suggest_on_error=True`, which adds "maybe you meant …?" hints for mistyped choices and subcommand names.

In [ ]:
helpful = argparse.ArgumentParser(prog="report", suggest_on_error=True, color=False)
helpful.add_argument("--format", choices=["table", "json"])
try_parse(helpful, ["--format", "jsn"])

⚠️ **Gotcha:** by default `argparse` accepts **unambiguous prefixes** of long options: `--lim 5` works as `--limit 5`. That is convenient, but renaming or adding an option later can break someone's scripts. Pass `allow_abbrev=False` to turn it off.

In [ ]:
print(parser.parse_args(["data.csv", "--lim", "5"]).limit)        # 5 — prefix accepted

## Subcommands

Tools like `git` or `dotnet` have subcommands (`dotnet build`, `dotnet test`), each with its own options. `add_subparsers()` returns an object whose `add_parser()` creates a nested parser per command. A common pattern is `set_defaults(func=...)` so each subcommand carries its handler.

```csharp
// C# — System.CommandLine 2.0
Command add = new("add", "Add a task");
Argument<string> titleArg = new("title");
add.Arguments.Add(titleArg);
add.SetAction(parseResult => Console.WriteLine($"added {parseResult.GetValue(titleArg)}"));
root.Subcommands.Add(add);
return root.Parse(args).Invoke();
```

In [ ]:
def cmd_add(args: argparse.Namespace) -> int:
    print(f"added {args.title!r} with priority {args.priority}")
    return 0

def cmd_list(args: argparse.Namespace) -> int:
    print(f"listing tasks (done={args.done})")
    return 0

parser = argparse.ArgumentParser(prog="tasks", color=False)
subcommands = parser.add_subparsers(dest="command", required=True, title="commands")

add = subcommands.add_parser("add", help="add a task")
add.add_argument("title")
add.add_argument("-p", "--priority", type=int, choices=range(1, 4), default=2)
add.set_defaults(func=cmd_add)

list_cmd = subcommands.add_parser("list", help="list tasks")
list_cmd.add_argument("--done", action="store_true")
list_cmd.set_defaults(func=cmd_list)

In [ ]:
for argv in (["add", "buy milk", "-p", "1"], ["list", "--done"]):
    args = parser.parse_args(argv)
    print(args.command, "→", end=" ")
    args.func(args)                             # dispatch to the handler

try_parse(parser, ["remove", "x"])              # unknown subcommand
parser.print_help()

## Putting it together: a testable `main`

The idiomatic script shape takes an optional `argv` list, so tests (and notebooks) can call it directly while the real entry point uses `sys.argv`:

```python
def main(argv: list[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)     # None → sys.argv[1:]
    return args.func(args)

if __name__ == "__main__":
    raise SystemExit(main())
```

In [ ]:
def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="greet", color=False)
    p.add_argument("name")
    p.add_argument("--times", type=int, default=1)
    p.add_argument("--shout", action="store_true")
    return p

def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    message = f"hello, {args.name}"
    for _ in range(args.times):
        print(message.upper() if args.shout else message)
    return 0

print(main(["alice", "--times", "2"]))
print(main(["bob", "--shout"]))

**Tip:** `vars(args)` turns the `Namespace` into a plain `dict` — handy for logging the effective configuration or passing options as `**kwargs`. `parse_known_args()` returns `(namespace, leftover_args)` instead of failing on unknown arguments, useful when forwarding the rest to another tool.

In [ ]:
args = build_parser().parse_args(["carol", "--times", "3"])
print(vars(args))

known, rest = build_parser().parse_known_args(["dave", "--shout", "--extra", "x"])
print(known, rest)

## Summary

| Task | C# / .NET (`System.CommandLine`) | Python (`argparse`) |
|---|---|---|
| Create parser | `new RootCommand("desc")` | `ArgumentParser(description="desc")` |
| Positional | `new Argument<string>("input")` | `add_argument("input")` |
| Option with type | `new Option<int>("--limit", "-n")` | `add_argument("-n", "--limit", type=int)` |
| Default | `DefaultValueFactory = _ => 10` | `default=10` |
| Allowed values | `AcceptOnlyFromAmong("csv", "json")` | `choices=["csv", "json"]` |
| Boolean switch | `Option<bool>` | `action="store_true"` |
| `--x` / `--no-x` | — | `action=argparse.BooleanOptionalAction` |
| Many values | `AllowMultipleArgumentsPerToken` | `nargs="+"` / `action="append"` |
| Subcommands | `root.Subcommands.Add(new Command("add"))` | `add_subparsers().add_parser("add")` |
| Handler | `command.SetAction(...)` | `set_defaults(func=handler)` |
| Parse | `root.Parse(args)` | `parser.parse_args(argv)` |
| Read value | `result.GetValue(option)` | `args.limit` |
| Errors | `result.Errors` | message on stderr + `SystemExit(2)` (or `exit_on_error=False`) |
| Help | automatic `-h` / `--help` | automatic `-h` / `--help`, `print_help()` |

## Exercises

1. Build a parser for `resize IMAGE --width W [--height H] [--keep-aspect | --no-keep-aspect]` where width is a required positive integer option, height is optional, and keep-aspect defaults to `True`. Parse `["cat.png", "--width", "800"]` and `["cat.png", "--width", "800", "--height", "600", "--no-keep-aspect"]`.
2. Using `try_parse`-style code (`redirect_stderr` + `except SystemExit`), show the error messages for `["cat.png"]` (missing `--width`) and `["cat.png", "--width", "-5"]`.
3. Create a `calc` parser with two subcommands, `add` and `mul`, each taking one or more numbers (`nargs="+"`, `type=float`), and use `set_defaults(func=...)` so that `["mul", "2", "3", "4"]` prints `24.0`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import argparse
import contextlib
import io
import math

# 1 — required option with a validating type, plus BooleanOptionalAction
def positive_int(text: str) -> int:
    value = int(text)
    if value <= 0:
        raise argparse.ArgumentTypeError(f"{value} is not > 0")
    return value

resize = argparse.ArgumentParser(prog="resize", color=False)
resize.add_argument("image")
resize.add_argument("--width", type=positive_int, required=True)
resize.add_argument("--height", type=positive_int)
resize.add_argument("--keep-aspect", action=argparse.BooleanOptionalAction, default=True)

print(resize.parse_args(["cat.png", "--width", "800"]))
print(resize.parse_args(["cat.png", "--width", "800", "--height", "600", "--no-keep-aspect"]))

# 2 — capture stderr, catch SystemExit
for argv in (["cat.png"], ["cat.png", "--width", "-5"]):
    err = io.StringIO()
    try:
        with contextlib.redirect_stderr(err):
            resize.parse_args(argv)
    except SystemExit as e:
        print(e.code, err.getvalue().strip().splitlines()[-1])

# 3 — subcommands carry their handler via set_defaults
calc = argparse.ArgumentParser(prog="calc", color=False)
sub = calc.add_subparsers(dest="command", required=True)
add = sub.add_parser("add")
add.add_argument("numbers", nargs="+", type=float)
add.set_defaults(func=lambda a: print(sum(a.numbers)))
mul = sub.add_parser("mul")
mul.add_argument("numbers", nargs="+", type=float)
mul.set_defaults(func=lambda a: print(math.prod(a.numbers)))

for argv in (["add", "1", "2.5"], ["mul", "2", "3", "4"]):
    args = calc.parse_args(argv)
    args.func(args)        # 3.5, then 24.0
```

</details>